# LLM Distillation / Fine-Tuning Demo

This notebook walks through the full pipeline for fine-tuning a small LLM on a
**single-file HTML website generation** task using **Unsloth** + **HuggingFace TRL**.

The workflow is:
1. **Generate training data** — query a powerful model via OpenRouter to get high-quality HTML examples.
2. **Evaluate the base model** — run the small model before fine-tuning on the eval test cases.
3. **Fine-tune** — apply QLoRA with Unsloth.
4. **Evaluate the fine-tuned model** — compare before / after quality in the browser.

Each pipeline step is driven by a **function imported directly from the corresponding
script file**, so all output appears live inside the cell rather than in a subprocess.
Training prompts and evaluation test cases are stored in **`prompts.json`**.

---
**Hardware:** RTX 2080 Ti (22 GB VRAM) — the defaults are tuned for this card.  
**Environment:** run this notebook inside the `distill_dev` Docker container started with:
```bash
sudo docker compose -f docker-compose-distill.yml up
```
Then open `http://localhost:8888`.

## 0. Configuration

In [2]:
import os, sys, json

# Make the distillation scripts importable when the notebook is opened from
# any working directory (e.g. the repo root inside the Docker container).
SCRIPT_DIR = os.path.dirname(os.path.abspath("distillation_demo.ipynb"))
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# Helps reduce CUDA fragmentation-related OOMs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# -----------------------------------------------------------------------
# Set your keys here OR export them as environment variables before
# starting the container.
# -----------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "<your-openrouter-key>")
HF_TOKEN          = os.getenv("HF_TOKEN", "")   # optional - only needed to push to Hub

# Local llama.cpp-compatible server URL (when using provider="local").
# Defaults to http://localhost:8080 but you can set LLAMA_CPP_SERVER_URL in your env.
LLAMA_CPP_SERVER_URL = os.getenv("LLAMA_CPP_SERVER_URL", "http://host.docker.internal:1337")

# Teacher model used to generate training data (via OpenRouter or local server)
# When using provider="local", this should match a model name served by your local server.
TEACHER_MODEL = "Qwen3.5-35B-A3B-UD-IQ4_NL.gguf"
# Alternatively: "mistralai/mistral-large", "openai/gpt-4o-mini", etc.

# Small student model to fine-tune
STUDENT_MODEL = "ibm-granite/granite-4.0-h-1b"
# Alternatives: "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#               "ibm-granite/granite-3.1-2b-instruct"

# Prompts file - contains training topics, instruction templates, and eval test cases
PROMPTS_FILE = os.path.join(SCRIPT_DIR, "prompts.json")

# Paths
DATASET_PATH   = os.path.join(SCRIPT_DIR, "website_dataset.jsonl")
ADAPTER_DIR    = os.path.join(SCRIPT_DIR, "outputs", "lora_model")
EVAL_BASE_DIR  = os.path.join(SCRIPT_DIR, "eval_results", "base")
EVAL_FT_DIR    = os.path.join(SCRIPT_DIR, "eval_results", "finetuned")

GGUF_DIR       = os.path.join(SCRIPT_DIR, "outputs", "gguf_model")
# GGUF quantization: q4_k_m (default), q5_k_m, q8_0, f16, q4_0, q5_0, q2_k, q3_k_m
GGUF_QUANT     = "q4_k_m"

N_SAMPLES      = 200   # number of training examples to generate
EPOCHS         = 1
BATCH_SIZE     = 1
GRAD_ACCUM     = 8     # effective batch = 1*8 = 8

# Memory-safe defaults for Granite on 22 GB VRAM.
TRAIN_MAX_SEQ_LENGTH = 1024
TRAIN_PACKING = False

print("Configuration loaded.")
print(f"  Script directory      : {SCRIPT_DIR}")
print(f"  Prompts file          : {PROMPTS_FILE}")
print(f"  LLAMA_CPP_SERVER_URL  : {LLAMA_CPP_SERVER_URL}")
print(f"  PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")
print(f"  Train max_seq_length  : {TRAIN_MAX_SEQ_LENGTH}")
print(f"  Train packing         : {TRAIN_PACKING}")

Configuration loaded.
  Script directory      : /code
  Prompts file          : /code/prompts.json
  LLAMA_CPP_SERVER_URL  : http://host.docker.internal:1337
  PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True
  Train max_seq_length  : 1024
  Train packing         : False


## 0b. Inspect Prompts File

All training topics, instruction templates, and evaluation test cases live in
`prompts.json`. Edit that file to add new topics or change eval cases without
touching any Python script.

In [3]:
with open(PROMPTS_FILE) as fh:
    prompts = json.load(fh)

print(f"Training topics      : {len(prompts['training_topics'])}")
print(f"Instruction templates: {len(prompts['instruction_templates'])}")
print(f"Eval test cases      : {len(prompts['eval_test_cases'])}")

print("\n--- First 5 training topics ---")
for t in prompts['training_topics'][:5]:
    print(f"  · {t}")

print("\n--- Eval test case IDs ---")
for tc in prompts['eval_test_cases']:
    print(f"  · {tc['id']}: {tc['instruction'][:80]}...")

Training topics      : 80
Instruction templates: 8
Eval test cases      : 8

--- First 5 training topics ---
  · a simple todo list app with add, complete, and delete functionality
  · a Pomodoro productivity timer with configurable work and break cycles
  · a kanban board with three columns: To Do, In Progress, and Done
  · a daily schedule planner with hourly time slots from 6 AM to midnight
  · a goal tracking page with labelled progress bars and percentage complete

--- Eval test case IDs ---
  · weather_chart: Build a self-contained HTML page for a 7-day weather forecast with SVG bar chart...
  · pixel_art_editor: Create a single-file HTML website: a pixel art editor with a 16x16 drawing grid,...
  · chat_ui: Write a complete single-file HTML + CSS + JS website that implements a chat appl...
  · spreadsheet_grid: Generate a responsive single-file HTML website for a mini spreadsheet with edita...
  · saas_landing: Produce a polished, self-contained HTML file that acts as an animate

## 1. Generate Training Data

We call a capable teacher model via **OpenRouter** to produce high-quality
single-file HTML examples across a variety of website types.  
Each record is saved as a JSON line: `{instruction, input, output}`.  
Topics and templates are read from `prompts.json`.

The cell imports `generate_dataset` directly from `generate_training_data.py`
so all progress output appears live in the cell.

In [8]:
from generate_training_data import generate_dataset

generate_dataset(
    api_key=HF_TOKEN,
    model=TEACHER_MODEL,
    n_samples=N_SAMPLES,
    output=DATASET_PATH,
    prompts_file=PROMPTS_FILE,
    provider="local",
    local_server_url=LLAMA_CPP_SERVER_URL,
    store_mode="file_path",
    html_output_dir=os.path.join(SCRIPT_DIR, "html_outputs")
)

Loaded 80 topics and 8 templates from /code/prompts.json.
Resuming — 3 samples already in /code/website_dataset.jsonl.


Generating: 100%|██████████| 200/200 [1:28:33<00:00, 26.57s/it] 

Done. 80 samples saved to /code/website_dataset.jsonl.


In [4]:
# Preview a few samples
with open(DATASET_PATH) as fh:
    lines = fh.readlines()
print(f"Total samples: {len(lines)}")
sample = json.loads(lines[0])
print("\n--- Instruction ---")
print(sample["instruction"])
print("\n--- Output (first 500 chars) ---")
print(sample["output"][:500])

Total samples: 80

--- Instruction ---
Create a single-file HTML website: a simple todo list app with add, complete, and delete functionality.

--- Output (first 500 chars) ---
<think>
We are building a simple todo list app with add, complete, and delete functionality.
 The app will have:
   - An input field to add a new task.
   - A button to add the task.
   - Each task in the list will have:
        * The task text.
        * A button to mark as complete (which might change the style of the task).
        * A button to delete the task.

 We'll use inline CSS for styling and inline JavaScript for the functionality.

 Steps:
   1. Structure:
        - A container for 


## 2. Evaluate the Base Model (Before Fine-Tuning)

We run the student model on the eval test cases defined in `prompts.json` and
save each result as an HTML file. Open them in a browser to visually inspect quality.
The eval prompts are intentionally **different** from the training topics so we
measure genuine generalisation, not memorisation.

The cell imports `evaluate` directly from `evaluate_model.py` so model loading
progress and per-case generation output appears live in the cell.

In [10]:
from evaluate_model import evaluate

base_eval_results = evaluate(
    model_name=STUDENT_MODEL,
    output_dir=EVAL_BASE_DIR,
    prompts_file=PROMPTS_FILE,
)

Loaded 8 eval test cases from /code/prompts.json.
Loading model: ibm-granite/granite-4.0-h-1b
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Granitemoehybrid patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 2080 Ti. Num GPUs = 1. Max memory: 21.484 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[weather_chart] Generating …
  Saved to /code/eval_results/base/weather_chart.html  (1524 chars)

[pixel_art_editor] Generating …
  Saved to /code/eval_results/base/pixel_art_editor.html  (2322 chars)

[chat_ui] Generating …
  Saved to /code/eval_results/base/chat_ui.html  (4885 chars)

[spreadsheet_grid] Generating …
  Saved to /code/eval_results/base/spreadsheet_grid.html  (3164 chars)

[saas_landing] Generating …
  Saved to /code/eval_results/base/saas_landing.html  (4319 chars)

[code_editor_ui] Generating …
  Saved to /code/eval_results/base/code_editor_ui.html  (1161 chars)

[habit_heatmap] Generating …
  Saved to /code/eval_results/base/habit_heatmap.html  (1840 chars)

[budget_planner] Generating …
  Saved to /code/eval_results/base/budget_planner.html  (3465 chars)

--- Evaluation Summary ---
  ✓ weather_chart         chars= 1524  html=True  body=True  script/style=True
  ✓ pixel_art_editor      chars= 2322  html=True  body=True  script/style=True
  ✓ chat_ui               ch

In [11]:
# Display the base-model summary (uses the return value from the cell above)
for r in base_eval_results:
    status = "✓" if (r["starts_with_html"] and r["has_body"]) else "✗"
    print(f"{status} {r['id']:24s}  chars={r['char_count']:5d}")

✓ weather_chart             chars= 1524
✓ pixel_art_editor          chars= 2322
✓ chat_ui                   chars= 4885
✓ spreadsheet_grid          chars= 3164
✓ saas_landing              chars= 4319
✓ code_editor_ui            chars= 1161
✓ habit_heatmap             chars= 1840
✓ budget_planner            chars= 3465


In [ ]:
# Render the first eval result inline
from IPython.display import IFrame
first_eval_id = prompts['eval_test_cases'][0]['id']
IFrame(src=os.path.join(EVAL_BASE_DIR, f"{first_eval_id}.html"), width="100%", height=500)

## 3. Fine-Tune with Unsloth

We apply **QLoRA** (4-bit quantised LoRA) using Unsloth's optimised kernels.
Expected training time on an RTX 2080 Ti: ~10–20 minutes for 200 samples × 3 epochs.

The cell imports `train` directly from `train_model.py` so all training logs
(loss, learning rate, etc.) appear live in the cell.

In [5]:
# Check whether TRAIN_MAX_SEQ_LENGTH will truncate training samples.
# Run this after dataset generation and before training.
from train_model import load_jsonl, format_sample
from transformers import AutoTokenizer
import os
import statistics

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. Run the data generation cell first."
    )

print(f"Loading tokenizer for: {STUDENT_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL,
    use_fast=True,
    trust_remote_code=True,
    token=HF_TOKEN or None,
    local_files_only=False,
    )

records = load_jsonl(DATASET_PATH)
length_rows = []

for idx, record in enumerate(records):
    text = format_sample(record, tokenizer)["text"]
    n_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    length_rows.append(
        {
            "index": idx,
            "tokens": n_tokens,
            "instruction": record.get("instruction", "")[:120],
        }
    )

token_counts = [row["tokens"] for row in length_rows]
limit = TRAIN_MAX_SEQ_LENGTH
over_limit = [row for row in length_rows if row["tokens"] > limit]

def percentile(values, p):
    if not values:
        return 0
    # Nearest-rank percentile to keep dependencies minimal
    k = max(1, round((p / 100) * len(values)))
    return sorted(values)[k - 1]

print("\n=== Token Length Audit ===")
print(f"Samples                 : {len(token_counts)}")
print(f"Train max_seq_length    : {limit}")
print(f"Min / Median / Mean     : {min(token_counts)} / {int(statistics.median(token_counts))} / {int(statistics.mean(token_counts))}")
print(
    f"P90 / P95 / P99 / Max   : "
    f"{percentile(token_counts, 90)} / "
    f"{percentile(token_counts, 95)} / "
    f"{percentile(token_counts, 99)} / "
    f"{max(token_counts)}"
)

pct = (len(over_limit) / len(token_counts) * 100) if token_counts else 0.0
print(f"Would be truncated      : {len(over_limit)} ({pct:.2f}%)")

if over_limit:
    print("\nTop 10 longest samples (likely truncation risk):")
    for row in sorted(over_limit, key=lambda x: x["tokens"], reverse=True)[:10]:
        print(
            f"  idx={row['index']:4d}  tokens={row['tokens']:5d}  "
            f"instruction={row['instruction']!r}"
        )
else:
    print("No samples exceed TRAIN_MAX_SEQ_LENGTH.")

suggested = percentile(token_counts, 95)
print(
    f"\nSuggestion: set TRAIN_MAX_SEQ_LENGTH around P95 ({suggested}) "
    "for a balance between context coverage and VRAM usage."
)

/code/train_model.py:41: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, is_bfloat16_supported


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading tokenizer for: ibm-granite/granite-4.0-h-1b


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]


=== Token Length Audit ===
Samples                 : 80
Train max_seq_length    : 1024
Min / Median / Mean     : 1403 / 3063 / 3160
P90 / P95 / P99 / Max   : 4152 / 5354 / 6107 / 6963
Would be truncated      : 80 (100.00%)

Top 10 longest samples (likely truncation risk):
  idx=  39  tokens= 6963  instruction='Build a modern, responsive single-file HTML app for a personal portfolio page for a software engineer with projects and '
  idx=  46  tokens= 6107  instruction='Implement a landing page for a mobile app with hero section, feature highlights, and screenshots as a single self-contai'
  idx=  76  tokens= 5645  instruction='Produce a polished, self-contained HTML file that acts as a video streaming homepage UI with mock thumbnail cards and a '
  idx=  24  tokens= 5360  instruction='Create a single-file HTML website: a stock portfolio summary page with mock ticker symbols and gains/losses.'
  idx=  69  tokens= 5354  instruction='Design and code a single HTML file that serves as an em

In [7]:
# Optional audit: estimate token savings if teacher "thinking" traces are removed.
# This does NOT modify your dataset file; it only reports potential impact.
import re
import os
import statistics
from train_model import load_jsonl, format_sample
from transformers import AutoTokenizer

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. Run the data generation cell first."
    )

print(f"Loading tokenizer for: {STUDENT_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL,
    use_fast=True,
    trust_remote_code=True,
    token=HF_TOKEN or None,
    local_files_only=False,
    )

# Common reasoning/thinking wrappers seen across models/providers.
THINK_PATTERNS = [
    re.compile(r"<think>.*?</think>", re.IGNORECASE | re.DOTALL),
    re.compile(r"<thinking>.*?</thinking>", re.IGNORECASE | re.DOTALL),
    re.compile(r"```thinking\s*.*?```", re.IGNORECASE | re.DOTALL),
    re.compile(r"```reasoning\s*.*?```", re.IGNORECASE | re.DOTALL),
    re.compile(r"\[thinking\].*?\[/thinking\]", re.IGNORECASE | re.DOTALL),
    re.compile(r"\[reasoning\].*?\[/reasoning\]", re.IGNORECASE | re.DOTALL),
]

def strip_thinking(text: str) -> tuple[str, bool]:
    if text is None:
        return "", False
    cleaned = text
    changed = False
    for pattern in THINK_PATTERNS:
        new_text, n = pattern.subn("", cleaned)
        if n > 0:
            changed = True
            cleaned = new_text
    # Normalize extra blank lines after removal.
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    return cleaned, changed

def get_assistant_text(record: dict) -> str:
    output = record.get("output")
    if output is not None:
        return output
    output_file = record.get("output_file")
    if output_file:
        if not os.path.exists(output_file):
            raise FileNotFoundError(f"Referenced output_file does not exist: {output_file}")
        with open(output_file, "r", encoding="utf-8") as fh:
            return fh.read()
    return ""

records = load_jsonl(DATASET_PATH)
base_lengths = []
clean_lengths = []
changed_rows = []
clean_rows = []

for idx, record in enumerate(records):
    base_text = format_sample(record, tokenizer)["text"]
    base_tokens = len(tokenizer(base_text, add_special_tokens=False)["input_ids"])

    assistant_text = get_assistant_text(record)
    cleaned_assistant, changed = strip_thinking(assistant_text)

    clean_record = dict(record)
    if "output" in clean_record:
        clean_record["output"] = cleaned_assistant
    elif "output_file" in clean_record:
        # For analysis only: inject cleaned output inline instead of file path.
        clean_record["output"] = cleaned_assistant
        clean_record.pop("output_file", None)

    clean_text = format_sample(clean_record, tokenizer)["text"]
    clean_tokens = len(tokenizer(clean_text, add_special_tokens=False)["input_ids"])

    base_lengths.append(base_tokens)
    clean_lengths.append(clean_tokens)
    clean_rows.append(
        {
            "index": idx,
            "tokens": clean_tokens,
            "before": base_tokens,
            "delta": base_tokens - clean_tokens,
            "instruction": record.get("instruction", "")[:120],
        }
    )

    if changed:
        changed_rows.append(
            {
                "index": idx,
                "before": base_tokens,
                "after": clean_tokens,
                "delta": base_tokens - clean_tokens,
                "instruction": record.get("instruction", "")[:120],
            }
        )

limit = TRAIN_MAX_SEQ_LENGTH
base_over = sum(1 for x in base_lengths if x > limit)
clean_over = sum(1 for x in clean_lengths if x > limit)
delta_tokens = [b - c for b, c in zip(base_lengths, clean_lengths)]

print("\n=== Thinking-Token Removal Audit ===")
print(f"Samples analyzed                : {len(records)}")
print(f"Rows with detected thinking tags: {len(changed_rows)}")
print(f"Train max_seq_length            : {limit}")
print(
    f"Mean tokens (before -> after)   : "
    f"{int(statistics.mean(base_lengths))} -> {int(statistics.mean(clean_lengths))}"
)
print(
    f"Median tokens (before -> after) : "
    f"{int(statistics.median(base_lengths))} -> {int(statistics.median(clean_lengths))}"
)
print(
    f"Max tokens (before -> after)    : "
    f"{max(base_lengths)} -> {max(clean_lengths)}"
)
print(
    f"Would truncate (before -> after): "
    f"{base_over} -> {clean_over} "
    f"({(base_over/len(records)*100):.2f}% -> {(clean_over/len(records)*100):.2f}%)"
)
print(f"Mean token reduction per sample : {statistics.mean(delta_tokens):.2f}")

print("\nTop 3 token counts after removing thinking tokens:")
for row in sorted(clean_rows, key=lambda r: r["tokens"], reverse=True)[:3]:
    print(
        f"  idx={row['index']:4d}  after={row['tokens']:5d}  "
        f"before={row['before']:5d}  saved={row['delta']:5d}  "
        f"instruction={row['instruction']!r}"
    )

if changed_rows:
    print("\nTop 10 samples with biggest token savings:")
    for row in sorted(changed_rows, key=lambda r: r["delta"], reverse=True)[:10]:
        print(
            f"  idx={row['index']:4d}  before={row['before']:5d}  "
            f"after={row['after']:5d}  saved={row['delta']:5d}  "
            f"instruction={row['instruction']!r}"
        )
else:
    print("\nNo common thinking-tag patterns were detected in the dataset output.")

print("\nIf the after-metrics look better, we can add a cell to write a cleaned dataset JSONL for training.")

Loading tokenizer for: ibm-granite/granite-4.0-h-1b

=== Thinking-Token Removal Audit ===
Samples analyzed                : 80
Rows with detected thinking tags: 3
Train max_seq_length            : 1024
Mean tokens (before -> after)   : 3160 -> 3125
Median tokens (before -> after) : 3063 -> 2894
Max tokens (before -> after)    : 6963 -> 6963
Would truncate (before -> after): 80 -> 80 (100.00% -> 100.00%)
Mean token reduction per sample : 34.76

Top 3 token counts after removing thinking tokens:
  idx=  39  after= 6963  before= 6963  saved=    0  instruction='Build a modern, responsive single-file HTML app for a personal portfolio page for a software engineer with projects and '
  idx=  46  after= 6107  before= 6107  saved=    0  instruction='Implement a landing page for a mobile app with hero section, feature highlights, and screenshots as a single self-contai'
  idx=  76  after= 5645  before= 5645  saved=    0  instruction='Produce a polished, self-contained HTML file that acts as a vi

In [ ]:
import importlib
import train_model
importlib.reload(train_model)

train_model.train(
    model_name=STUDENT_MODEL,
    dataset_path=DATASET_PATH,
    output_dir=ADAPTER_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    max_seq_length=TRAIN_MAX_SEQ_LENGTH,
    packing=TRAIN_PACKING,
)

Loading base model: ibm-granite/granite-4.0-h-1b
==((====))==  Unsloth 2026.2.1: Fast Granitemoehybrid patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 2080 Ti. Num GPUs = 1. Max memory: 21.484 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Making `model.base_model.model.model` require gradients
trainable params: 655,360 || all params: 1,462,193,728 || trainable%: 0.0448
Loaded 80 training samples.
Unsloth: Sample packing skipped (custom data collator detected).
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/80 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 655,360 of 1,462,193,728 (0.04% trained)


Starting training …
Unsloth: Will smartly offload gradients to save VRAM!


OutOfMemoryError: CUDA out of memory. Tried to allocate 30.00 GiB. GPU 0 has a total capacity of 21.48 GiB of which 17.66 GiB is free. Process 3672 has 212.00 MiB memory in use. Process 781523 has 3.03 GiB memory in use. Of the allocated memory 2.65 GiB is allocated by PyTorch, and 176.50 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 3b. (Optional) Export Fine-Tuned Adapter to GGUF

Unsloth can merge the LoRA adapter into the base model and export the result
as a **GGUF** file, ready for local inference with **llama.cpp** or **Ollama**.

`save_gguf()` is a separate function in `train_model.py` that:
1. Validates the adapter directory and (if supplied) a local base-model path.
2. Loads the base model + LoRA adapter.
3. Calls Unsloth's `save_pretrained_gguf` with the chosen quantization.

Supported quantization methods: `q4_k_m` *(default — recommended)*, `q5_k_m`,
`q8_0`, `f16`, `q4_0`, `q5_0`, `q2_k`, `q3_k_m`.

In [ ]:
from train_model import save_gguf

gguf_path = save_gguf(
    adapter_dir=ADAPTER_DIR,
    output_dir=GGUF_DIR,
    quantization_method=GGUF_QUANT,
    model_name=STUDENT_MODEL,
)
print(f"GGUF export complete: {gguf_path}")

## 4. Evaluate the Fine-Tuned Model

Same eval test cases from `prompts.json`, now run with the LoRA adapter applied.

In [ ]:
from evaluate_model import evaluate

ft_eval_results = evaluate(
    model_name=STUDENT_MODEL,
    adapter_path=ADAPTER_DIR,
    output_dir=EVAL_FT_DIR,
    prompts_file=PROMPTS_FILE,
)

## 5. Before / After Comparison

In [ ]:
# Build lookup dicts from the results returned by the evaluate() calls above.
# If you restarted the kernel, load from the saved summary files instead:
#   with open(os.path.join(EVAL_BASE_DIR, "summary.json")) as fh:
#       base_eval_results = json.load(fh)
#   with open(os.path.join(EVAL_FT_DIR, "summary.json")) as fh:
#       ft_eval_results = json.load(fh)

base_by_id = {r["id"]: r for r in base_eval_results}
ft_by_id   = {r["id"]: r for r in ft_eval_results}

print(f"{'Test Case':<26} {'Base chars':>12} {'FT chars':>10} {'Base ✓':>8} {'FT ✓':>6}")
print("-" * 66)
for tc_id in base_by_id:
    b = base_by_id[tc_id]
    f = ft_by_id.get(tc_id, {})
    b_ok = "✓" if (b.get("starts_with_html") and b.get("has_body")) else "✗"
    f_ok = "✓" if (f.get("starts_with_html") and f.get("has_body")) else "✗"
    print(f"{tc_id:<26} {b['char_count']:>12} {f.get('char_count', 0):>10} {b_ok:>8} {f_ok:>6}")

In [ ]:
# Side-by-side comparison — change first_eval_id to any eval case id you like
from IPython.display import display, HTML

display(HTML(f"<h3>Base model — {first_eval_id}</h3>"))
display(IFrame(src=os.path.join(EVAL_BASE_DIR, f"{first_eval_id}.html"), width="100%", height=400))

display(HTML(f"<h3>Fine-tuned model — {first_eval_id}</h3>"))
display(IFrame(src=os.path.join(EVAL_FT_DIR, f"{first_eval_id}.html"), width="100%", height=400))

## 6. (Optional) Interactive Inference

In [ ]:
from evaluate_model import load_model_and_tokenizer, generate_html, extract_html, SYSTEM_PROMPT

inf_model, inf_tokenizer = load_model_and_tokenizer(STUDENT_MODEL, ADAPTER_DIR)
print("Model ready.")

In [ ]:
# Try your own prompt!
raw = generate_html(
    inf_model,
    inf_tokenizer,
    "Create a single-file HTML website: a dark-mode toggle demo page.",
    max_new_tokens=2048,
)
html = extract_html(raw)
print(html[:800])

In [ ]:
# Render inline
from IPython.display import display, HTML as _HTML
display(_HTML(html))

## 7. (Optional) Push Adapter to HuggingFace Hub

In [ ]:
# Uncomment and fill in your HF repo id to publish the adapter
# HF_REPO = "your-username/llama-3.2-3b-website-lora"
# from huggingface_hub import login
# login(token=HF_TOKEN)
# from train_model import train
# train(
#     model_name=STUDENT_MODEL,
#     dataset_path=DATASET_PATH,
#     output_dir=ADAPTER_DIR,
#     push_to_hub=HF_REPO,
# )
# print(f"Pushed to https://huggingface.co/{HF_REPO}")